## 03_spatial_aggregation
  Assigns OSM features to NUTS3 regions by spatial join and merges them with the Eurostat tables.
  Input: data/interim/osm/, GISCO NUTS3 boundaries, Eurostat TSV files.
  Output: data/processed/nuts3_master_table.parquet (115 regions x 10 years), nuts2_ai_adoption_table.parquet.

In [ ]:
import sys
sys.path.insert(0, "..")  # ha a notebook a notebooks/ mappából fut

import gc
import geopandas as gpd
import pandas as pd
import pyarrow.parquet as pq

from src.config import (
    PROJECT_ROOT,
    RAW_NUTS3_DIR,
    RAW_NUTS2_DIR,
    RAW_EUROSTAT_DIR,
    INTERIM_OSM_DIR,
    INTERIM_AGGREGATED_DIR,
    PROCESSED_DIR,
    NUTS_SCALE,
    NUTS_YEAR,
    COUNTRY_CODES,
    EUROSTAT_SOURCES,
    EUROSTAT_YEARS_MAIN,
    NUTS3_MASTER_TABLE_PATH,
)
from src.spatial_join import aggregate_to_nuts3

In [ ]:
nuts3_filename = f"NUTS_RG_{NUTS_SCALE}_{NUTS_YEAR}_4326.geojson"
nuts2_filename = f"NUTS_RG_{NUTS_SCALE}_{NUTS_YEAR}_4326_LEVL_2.geojson"

nuts3_path = RAW_NUTS3_DIR / nuts3_filename
nuts2_path = RAW_NUTS2_DIR / nuts2_filename

assert nuts3_path.exists(), f"Nincs meg: {nuts3_path}"
assert nuts2_path.exists(), f"Nincs meg: {nuts2_path}"

nuts3_gdf = gpd.read_file(nuts3_path)
nuts2_gdf = gpd.read_file(nuts2_path)

# V4-szűrés: csak HU/PL/CZ/SK régiók
nuts3_gdf = nuts3_gdf[nuts3_gdf["CNTR_CODE"].isin([c.upper() for c in COUNTRY_CODES])].copy()
nuts2_gdf = nuts2_gdf[nuts2_gdf["CNTR_CODE"].isin([c.upper() for c in COUNTRY_CODES])].copy()

# nuts3_id oszlop létrehozása a spatial_join.py elvárása szerint (NUTS_ID -> nuts3_id)
nuts3_gdf = nuts3_gdf.rename(columns={"NUTS_ID": "nuts3_id"})
nuts2_gdf = nuts2_gdf.rename(columns={"NUTS_ID": "nuts2_id"})

print(f"NUTS3 régiók (V4): {len(nuts3_gdf)}")  # várt: 115
print(f"NUTS2 régiók (V4): {len(nuts2_gdf)}")  # várt: 37
print(nuts3_gdf.crs)

In [ ]:
def iter_tile_batches(path, columns=None, batch_rows: int = 200_000, crs="EPSG:4326"):
    """
    Egy OSM-csempét Arrow row-group-batch-enként ad vissza GeoDataFrame-ként,
    ahelyett hogy az egészet egyszerre olvasná be memóriába egy
    gpd.read_parquet()-tel.

    Miért kell ez a csempénkénti del+gc.collect() mellett is: a
    complete_ways-es 02-es újravágás után egyes csempék (elsősorban PL
    buildings/pois) extrém nagyra nőhettek (lásd pl. cz_buildings_tile01
    1.67M sorra nőtt) — a csempénkénti streamelés önmagában nem elég, ha
    MAGA egy csempe egyben már túl nagy egy lépésben beolvasva.

    A GeoParquet-fájlokban a geometria WKB bytes-ként van tárolva, ezért
    egy Arrow batch pandas DataFrame-jén a 'geometry' oszlopot explicit
    WKB -> shapely dekódolással kell GeoSeries-szé alakítani.
    """
    pf = pq.ParquetFile(path)
    n_rows = pf.metadata.num_rows
    if n_rows > batch_rows:
        n_batches = -(-n_rows // batch_rows)  # ceil division
        print(f"    {path.name}: {n_rows:,} sor -> {n_batches} batch ({batch_rows:,} sor/batch)")
    else:
        print(f"    {path.name}: {n_rows:,} sor (1 batch)")

    for record_batch in pf.iter_batches(batch_size=batch_rows, columns=columns):
        chunk = record_batch.to_pandas()
        if "geometry" in chunk.columns:
            chunk["geometry"] = gpd.GeoSeries.from_wkb(chunk["geometry"])
        yield gpd.GeoDataFrame(chunk, geometry="geometry", crs=crs)



In [ ]:
def load_country_layer(country_code: str, layer_name: str, dedup: bool = True, columns=None) -> gpd.GeoDataFrame:
    """
    Egy ország egy rétegének összes csempéjét (és PL esetén subtile-ját)
    beolvassa, összefűzi, és opcionálisan id-dedup-olja.

    country_code: 'hu' / 'pl' / 'cz' / 'sk'
    layer_name:   'network_driving' / 'network_walking' / 'buildings' / 'pois'
    dedup:        True esetén drop_duplicates(subset="id") összefűzés után.
                  Lásd tile_partitioning_notes.md: hossz-/terület-összegzésnél
                  (road_length_km, pedestrian_network_length_km,
                  building_footprint_sqm) dedup=False, darabszám-alapú
                  mutatóknál (building_count, poi_count) dedup=True.
    columns:      ha meg van adva, csak ezeket az oszlopokat olvassa be
                  diszkről (memóriatakarékosság nagy rétegeknél, pl.
                  network_walking-nál elég ['id', 'geometry']).
    """
    pattern = f"{country_code}_{layer_name}_tile*.parquet"
    tile_paths = sorted(INTERIM_OSM_DIR.glob(pattern))

    if not tile_paths:
        raise FileNotFoundError(
            f"Nincs csempefájl ehhez a mintához: {INTERIM_OSM_DIR / pattern}"
        )

    frames = []
    for path in tile_paths:
        gdf = gpd.read_parquet(path, columns=columns)
        if "id" not in gdf.columns:
            raise ValueError(f"Hiányzó 'id' oszlop: {path}")
        frames.append(gdf)

    combined = pd.concat(frames, ignore_index=True)
    combined = gpd.GeoDataFrame(combined, geometry="geometry", crs=frames[0].crs)
    del frames

    if dedup:
        before = len(combined)
        combined = combined.drop_duplicates(subset="id").reset_index(drop=True)
        after = len(combined)
        if before != after:
            print(f"{country_code}/{layer_name}: {before - after} duplikátum eltávolítva ({before} -> {after})")

    return combined


def sjoin_and_count(
    layer_gdf: gpd.GeoDataFrame,
    nuts3_gdf: gpd.GeoDataFrame,
    country_code: str,
    out_col: str,
    groupby_col=None,
) -> pd.DataFrame:
    """
    Pontszerű vagy poligon-réteg (pl. buildings, pois) darabszám-alapú
    aggregálása NUTS3 szintre, sjoin(predicate="within") használatával
    (nem overlay — lásd 03-as memóriaprobléma, mszell-jegyzet 1. pont).

    country_code: 'hu' / 'pl' / 'cz' / 'sk' — a sjoin után csak az adott
    ország saját nuts3_id-jeire szűr, mert egy ország OSM-kivonata
    tartalmazhat szomszédos országba eső elemeket is (pl. hu-latest.osm.pbf
    tartalmazott SK-régióba eső épületeket).
    """
    joined = layer_gdf.sjoin(
        nuts3_gdf[["nuts3_id", "geometry"]],
        predicate="within",
        how="left",
    )
    joined = joined[joined["nuts3_id"].str.startswith(country_code.upper())]

    if groupby_col:
        grouped = joined.groupby(["nuts3_id", groupby_col]).size().unstack(fill_value=0)
        grouped.columns = [f"{out_col}_{c}" for c in grouped.columns]
        result = grouped.reset_index()
    else:
        result = joined.groupby("nuts3_id").size().rename(out_col).reset_index()

    del joined
    return result


def streamed_length_aggregation(
    country_code: str,
    layer_name: str,
    nuts3_gdf: gpd.GeoDataFrame,
    out_col: str,
    batch_rows: int = 200_000,
) -> pd.DataFrame:
    """
    Hossz-alapú metrika (road_length_km, pedestrian_network_length_km)
    csempénkénti ÉS batch-enkénti, streamelt aggregálása — sem az egész
    országot, sem egyetlen (esetleg extrém nagy, complete_ways utáni)
    csempét nem tölti be egyszerre memóriába. Minden batch overlay-elve,
    részösszegzve, majd eldobva memóriából, mielőtt a következő jön —
    lásd iter_tile_batches().
    """
    pattern = f"{country_code}_{layer_name}_tile*.parquet"
    tile_paths = sorted(INTERIM_OSM_DIR.glob(pattern))

    if not tile_paths:
        raise FileNotFoundError(f"Nincs csempefájl ehhez: {INTERIM_OSM_DIR / pattern}")

    country_nuts3 = nuts3_gdf[nuts3_gdf["nuts3_id"].str.startswith(country_code.upper())]

    partial_results = []
    for path in tile_paths:
        for batch_gdf in iter_tile_batches(path, columns=["id", "geometry"], batch_rows=batch_rows, crs=nuts3_gdf.crs):
            batch_result = aggregate_to_nuts3(
                batch_gdf,
                country_nuts3,
                {out_col: {"method": "sum_length_km"}},
            )
            partial_results.append(batch_result)
            del batch_gdf, batch_result
        gc.collect()

    combined = pd.concat(partial_results, ignore_index=True)
    del partial_results
    final = combined.groupby("nuts3_id")[out_col].sum().reset_index()
    del combined
    return final

In [ ]:
def streamed_sjoin_count(
    country_code: str,
    layer_name: str,
    nuts3_gdf: gpd.GeoDataFrame,
    out_col: str,
    dedup: bool = True,
    batch_rows: int = 200_000,
) -> pd.DataFrame:
    """
    Darabszám-alapú metrika (building_count, poi_count) csempénkénti ÉS
    batch-enkénti, streamelt sjoin-aggregálása (lásd iter_tile_batches) —
    sem az egész országot, sem egyetlen csempét nem tölti be egyszerre
    memóriába.

    FONTOS ELTÉRÉS a korábbi (load_country_layer + sjoin_and_count) mintától:
    itt a dedup BATCH-ENKÉNT történik, nem az összefűzött teljes rétegen.
    Ez azt jelenti, hogy egy csempe- vagy batch-határon átnyúló duplikátum
    (azonos id, két külön batch-ben) NEM szűrődik ki. A HU-teszt szerint
    ez ~0.001% (45/3.8M) — elhanyagolható, de tudatosan vállalt pontatlanság
    a memóriabiztonságért cserébe.
    """
    pattern = f"{country_code}_{layer_name}_tile*.parquet"
    tile_paths = sorted(INTERIM_OSM_DIR.glob(pattern))
    if not tile_paths:
        raise FileNotFoundError(f"Nincs csempefájl ehhez: {INTERIM_OSM_DIR / pattern}")

    country_nuts3 = nuts3_gdf[nuts3_gdf["nuts3_id"].str.startswith(country_code.upper())]

    partial_results = []
    for path in tile_paths:
        for batch_gdf in iter_tile_batches(path, columns=["id", "geometry"], batch_rows=batch_rows, crs=nuts3_gdf.crs):
            if dedup:
                batch_gdf = batch_gdf.drop_duplicates(subset="id")

            joined = batch_gdf.sjoin(
                country_nuts3[["nuts3_id", "geometry"]],
                predicate="within",
                how="inner",
            )
            batch_result = joined.groupby("nuts3_id").size().rename(out_col).reset_index()
            partial_results.append(batch_result)

            del batch_gdf, joined, batch_result
        gc.collect()

    combined = pd.concat(partial_results, ignore_index=True)
    del partial_results
    final = combined.groupby("nuts3_id")[out_col].sum().reset_index()
    del combined
    return final

In [ ]:
pl_building_counts = streamed_sjoin_count("pl", "buildings", nuts3_gdf, "building_count", dedup=True)
print(pl_building_counts.shape)
pl_building_counts.head()

In [ ]:
def streamed_sjoin_area(
    country_code: str,
    layer_name: str,
    nuts3_gdf: gpd.GeoDataFrame,
    out_col: str,
    batch_rows: int = 200_000,
) -> pd.DataFrame:
    """
    Terület-alapú metrika (building_footprint_sqm) csempénkénti ÉS
    batch-enkénti, streamelt sjoin-aggregálása (lásd iter_tile_batches).
    Nincs dedup (lásd korábbi döntés: a csonkolt töredékek területe
    összegezve helyes eredményt ad).
    """
    pattern = f"{country_code}_{layer_name}_tile*.parquet"
    tile_paths = sorted(INTERIM_OSM_DIR.glob(pattern))
    if not tile_paths:
        raise FileNotFoundError(f"Nincs csempefájl ehhez: {INTERIM_OSM_DIR / pattern}")

    country_nuts3 = nuts3_gdf[nuts3_gdf["nuts3_id"].str.startswith(country_code.upper())]
    metric_crs = nuts3_gdf.estimate_utm_crs()

    partial_results = []
    for path in tile_paths:
        for batch_gdf in iter_tile_batches(path, columns=["id", "geometry"], batch_rows=batch_rows, crs=nuts3_gdf.crs):
            joined = batch_gdf.sjoin(
                country_nuts3[["nuts3_id", "geometry"]],
                predicate="within",
                how="inner",
            )
            joined = joined.to_crs(metric_crs)
            joined["area_sqm"] = joined.geometry.area

            batch_result = joined.groupby("nuts3_id")["area_sqm"].sum().rename(out_col).reset_index()
            partial_results.append(batch_result)

            del batch_gdf, joined, batch_result
        gc.collect()

    combined = pd.concat(partial_results, ignore_index=True)
    del partial_results
    final = combined.groupby("nuts3_id")[out_col].sum().reset_index()
    del combined
    return final

In [ ]:
pl_building_areas = streamed_sjoin_area("pl", "buildings", nuts3_gdf, "building_footprint_sqm")
print(pl_building_areas.shape)
pl_building_areas.head()

In [ ]:
def streamed_poi_aggregation(
    country_code: str,
    nuts3_gdf: gpd.GeoDataFrame,
) -> pd.DataFrame:
    """
    POI-réteg csempénkénti, streamelt sjoin-aggregálása: poi_count
    (összesített) + 5 innováció-proxy alkategória. Dedup csempénként
    (lásd streamed_sjoin_count megjegyzése a cross-tile duplikátumokról).
    """
    pattern = f"{country_code}_pois_tile*.parquet"
    tile_paths = sorted(INTERIM_OSM_DIR.glob(pattern))
    if not tile_paths:
        raise FileNotFoundError(f"Nincs csempefájl ehhez: {INTERIM_OSM_DIR / pattern}")

    country_nuts3 = nuts3_gdf[nuts3_gdf["nuts3_id"].str.startswith(country_code.upper())]

    partial_results = []
    for path in tile_paths:
        tile_gdf = gpd.read_parquet(path, columns=["id", "geometry", "amenity", "office"])
        tile_gdf = tile_gdf.drop_duplicates(subset="id")

        joined = tile_gdf.sjoin(
            country_nuts3[["nuts3_id", "geometry"]],
            predicate="within",
            how="inner",
        )

        joined["is_university"] = joined["amenity"].isin(["university", "college"])
        joined["is_research"]   = joined["amenity"] == "research_institute"
        joined["is_office"]     = joined["office"].notna()
        joined["is_coworking"]  = (
            (joined["office"] == "coworking") |
            (joined["amenity"] == "coworking_space")
        )
        joined["is_library"]    = joined["amenity"] == "library"

        tile_result = joined.groupby("nuts3_id").agg(
            poi_count=("id", "count"),
            poi_count_university=("is_university", "sum"),
            poi_count_research=("is_research", "sum"),
            poi_count_office=("is_office", "sum"),
            poi_count_coworking=("is_coworking", "sum"),
            poi_count_library=("is_library", "sum"),
        ).reset_index()

        partial_results.append(tile_result)
        del tile_gdf, joined, tile_result
        gc.collect()

    combined = pd.concat(partial_results, ignore_index=True)
    del partial_results
    final = combined.groupby("nuts3_id").sum().reset_index()
    del combined
    return final

In [ ]:
pl_poi_agg = streamed_poi_aggregation("pl", nuts3_gdf)
print(pl_poi_agg.shape)
pl_poi_agg.head()

In [ ]:
pl_road_lengths = streamed_length_aggregation("pl", "network_driving", nuts3_gdf, "road_length_km")
print(pl_road_lengths.shape)
pl_road_lengths.head()

In [ ]:
pl_pedestrian_lengths = streamed_length_aggregation("pl", "network_walking", nuts3_gdf, "pedestrian_network_length_km")
print(pl_pedestrian_lengths.shape)
pl_pedestrian_lengths.head()

In [ ]:
def streamed_poi_aggregation(
    country_code: str,
    nuts3_gdf: gpd.GeoDataFrame,
    batch_rows: int = 200_000,
) -> pd.DataFrame:
    """
    POI-réteg csempénkénti ÉS batch-enkénti, streamelt sjoin-aggregálása
    (lásd iter_tile_batches): poi_count (összesített) + 5 innováció-proxy
    alkategória.

    FONTOS: a pyrosm.get_pois() sémája országonként eltérhet (pl. SK
    pois rétegében nincs 'office' oszlop, mert egyszerűen nem fordult
    elő ilyen tag az adott ország OSM-adatában) — ezért csempénként
    ellenőrizzük, mely oszlopok léteznek, és hiányzókat NaN-nal pótoljuk.
    """
    pattern = f"{country_code}_pois_tile*.parquet"
    tile_paths = sorted(INTERIM_OSM_DIR.glob(pattern))
    if not tile_paths:
        raise FileNotFoundError(f"Nincs csempefájl ehhez: {INTERIM_OSM_DIR / pattern}")

    country_nuts3 = nuts3_gdf[nuts3_gdf["nuts3_id"].str.startswith(country_code.upper())]
    needed_cols = ["id", "geometry", "amenity", "office"]

    partial_results = []
    for path in tile_paths:
        available_cols = pq.read_schema(path).names
        cols_to_read = [c for c in needed_cols if c in available_cols]
        missing_cols = set(needed_cols) - set(cols_to_read)

        for batch_gdf in iter_tile_batches(path, columns=cols_to_read, batch_rows=batch_rows, crs=nuts3_gdf.crs):
            for missing_col in missing_cols:
                batch_gdf[missing_col] = pd.NA

            batch_gdf = batch_gdf.drop_duplicates(subset="id")

            joined = batch_gdf.sjoin(
                country_nuts3[["nuts3_id", "geometry"]],
                predicate="within",
                how="inner",
            )

            joined["is_university"] = joined["amenity"].isin(["university", "college"])
            joined["is_research"]   = joined["amenity"] == "research_institute"
            joined["is_office"]     = joined["office"].notna()
            joined["is_coworking"]  = (
                (joined["office"] == "coworking") |
                (joined["amenity"] == "coworking_space")
            )
            joined["is_library"]    = joined["amenity"] == "library"

            batch_result = joined.groupby("nuts3_id").agg(
                poi_count=("id", "count"),
                poi_count_university=("is_university", "sum"),
                poi_count_research=("is_research", "sum"),
                poi_count_office=("is_office", "sum"),
                poi_count_coworking=("is_coworking", "sum"),
                poi_count_library=("is_library", "sum"),
            ).reset_index()

            partial_results.append(batch_result)
            del batch_gdf, joined, batch_result
        gc.collect()

    combined = pd.concat(partial_results, ignore_index=True)
    del partial_results
    final = combined.groupby("nuts3_id").sum().reset_index()
    del combined
    return final

In [ ]:
def process_country(
    country_code: str,
    nuts3_gdf: gpd.GeoDataFrame,
    overwrite: bool = False,
    batch_rows: int = 200_000,
) -> pd.DataFrame:
    """
    Egy ország mind az 5 OSM-alapú metrikáját kiszámolja streamelt,
    csempénkénti ÉS batch-enkénti aggregálással, összefűzi, és lemezre
    menti.

    overwrite: False esetén, ha a kimeneti fájl már létezik, nem
    számol újra, hanem egyszerűen beolvassa és visszaadja azt. Ha a
    csempefájlok újabbak, mint a cache (pl. egy friss complete_ways-es
    02-es újravágás után), FIGYELMEZTETÉST ír ki, de nem számol
    automatikusan újra — a döntést a hívóra bízza.

    batch_rows: hány sort olvasson be egyszerre egy csempén belül
    (lásd iter_tile_batches) — csökkentsd, ha egy adott ország/réteg
    csempéi extrém nagyok (pl. PL buildings/pois complete_ways után).
    """
    out_path = INTERIM_AGGREGATED_DIR / f"nuts3_osm_metrics_{country_code}.parquet"

    if out_path.exists() and not overwrite:
        newest_tile_mtime = max(
            (p.stat().st_mtime for p in INTERIM_OSM_DIR.glob(f"{country_code}_*_tile*.parquet")),
            default=0,
        )
        cache_mtime = out_path.stat().st_mtime
        if newest_tile_mtime > cache_mtime:
            print(
                f"FIGYELEM: {country_code} cache ({out_path.name}) RÉGEBBI, mint a "
                f"csempefájlok -- valószínűleg a complete_ways-es 02-es újravágás "
                f"ELŐTTről származik (elavult adat). Töröld a cache-fájlt, vagy "
                f"hívd overwrite=True-val, ha friss adatot akarsz."
            )
        result = pd.read_parquet(out_path)
        print(f"{country_code}: már megvan -> {out_path.relative_to(PROJECT_ROOT)} ({result.shape})")
        return result

    building_counts = streamed_sjoin_count(country_code, "buildings", nuts3_gdf, "building_count", dedup=True, batch_rows=batch_rows)
    building_areas  = streamed_sjoin_area(country_code, "buildings", nuts3_gdf, "building_footprint_sqm", batch_rows=batch_rows)
    poi_agg         = streamed_poi_aggregation(country_code, nuts3_gdf, batch_rows=batch_rows)
    road_lengths    = streamed_length_aggregation(country_code, "network_driving", nuts3_gdf, "road_length_km", batch_rows=batch_rows)
    ped_lengths     = streamed_length_aggregation(country_code, "network_walking", nuts3_gdf, "pedestrian_network_length_km", batch_rows=batch_rows)

    result = (
        building_counts
        .merge(building_areas, on="nuts3_id", how="outer")
        .merge(poi_agg, on="nuts3_id", how="outer")
        .merge(road_lengths, on="nuts3_id", how="outer")
        .merge(ped_lengths, on="nuts3_id", how="outer")
    )

    result.to_parquet(out_path)
    print(f"{country_code}: mentve -> {out_path.relative_to(PROJECT_ROOT)} ({result.shape})")

    del building_counts, building_areas, poi_agg, road_lengths, ped_lengths
    gc.collect()

    return result

In [ ]:
result_hu = process_country("hu", nuts3_gdf, overwrite=False)
result_hu.head()

In [ ]:
result_sk = process_country("sk", nuts3_gdf, overwrite=False)
result_sk.head()

In [ ]:
result_cz = process_country("cz", nuts3_gdf, overwrite=False)
result_cz.head()

In [ ]:
for layer in ["buildings", "pois", "network_driving", "network_walking"]:
    tile_paths = sorted(INTERIM_OSM_DIR.glob(f"pl_{layer}_tile*.parquet"))
    print(f"--- pl / {layer} ({len(tile_paths)} csempe) ---")
    for p in tile_paths:
        n_rows = pq.ParquetFile(p).metadata.num_rows
        size_mb = p.stat().st_size / 1024**2
        print(f"  {p.name}: {n_rows:,} sor, {size_mb:,.1f} MB")
    print()


In [ ]:
result_pl = process_country("pl", nuts3_gdf, overwrite=False, batch_rows=100_000)
result_pl.head()

In [ ]:
osm_frames = []
for country_code in ["hu", "sk", "cz", "pl"]:
    path = INTERIM_AGGREGATED_DIR / f"nuts3_osm_metrics_{country_code}.parquet"
    osm_frames.append(pd.read_parquet(path))

nuts3_osm_metrics = pd.concat(osm_frames, ignore_index=True)
del osm_frames

print(nuts3_osm_metrics.shape)
nuts3_osm_metrics.head()

In [ ]:
print(nuts3_osm_metrics["nuts3_id"].str[:2].value_counts())

In [ ]:
nuts3_osm_metrics.to_parquet(INTERIM_AGGREGATED_DIR / "nuts3_osm_metrics_all.parquet")

In [ ]:
check_all = pd.read_parquet(INTERIM_AGGREGATED_DIR / "nuts3_osm_metrics_all.parquet")

print(check_all.shape)
print(check_all["nuts3_id"].str[:2].value_counts())
print(check_all["nuts3_id"].duplicated().sum())

In [ ]:
sample_path = RAW_EUROSTAT_DIR / "demo_r_pjangrp3.tsv"
with open(sample_path) as f:
    for _ in range(3):
        print(f.readline())

In [ ]:
def load_eurostat_variable(var_name: str) -> pd.DataFrame:
    spec = EUROSTAT_SOURCES[var_name]
    path = RAW_EUROSTAT_DIR / f"{spec['dataset_code']}.tsv"

    raw = pd.read_csv(path, sep="\t")
    dim_col = raw.columns[0]
    dim_names = dim_col.split("\\")[0].split(",")

    dims = raw[dim_col].str.split(",", expand=True)
    dims.columns = dim_names
    raw = pd.concat([dims, raw.drop(columns=[dim_col])], axis=1)
    raw.columns = [c.strip() for c in raw.columns]

    for key, val in spec["dim_filter"].items():
        raw = raw[raw[key] == val]

    if spec["geo_level"] == "nuts3":
        valid_codes = set(nuts3_gdf["nuts3_id"].unique())
    else:
        valid_codes = set(nuts2_gdf["nuts2_id"].unique())

    raw = raw[raw["geo"].isin(valid_codes)]

    # spec["years"] lehet egyetlen int (pl. EUROSTAT_YEAR_AI) vagy lista (EUROSTAT_YEARS_MAIN)
    spec_years = spec["years"]
    if isinstance(spec_years, int):
        spec_years = [spec_years]

    year_cols = [c for c in raw.columns if c.strip().isdigit() and int(c.strip()) in spec_years]

    long_df = raw.melt(id_vars=["geo"], value_vars=year_cols, var_name="year", value_name=var_name)

    long_df["year"] = long_df["year"].astype(int)
    long_df[var_name] = long_df[var_name].astype(str).str.replace(r"[^0-9.\-]", "", regex=True)
    long_df[var_name] = pd.to_numeric(long_df[var_name], errors="coerce")

    id_col = "nuts3_id" if spec["geo_level"] == "nuts3" else "nuts2_id"
    return long_df.rename(columns={"geo": id_col})

In [ ]:
pop_df = load_eurostat_variable("population")
print(pop_df.shape)
print(pop_df["nuts3_id"].nunique())

In [ ]:
ai_df = load_eurostat_variable("ai_adoption_rate")
print(ai_df.shape)
print(ai_df["nuts2_id"].nunique())
ai_df.head()

In [ ]:
nuts3_vars = [name for name, spec in EUROSTAT_SOURCES.items() if spec["geo_level"] == "nuts3"]
nuts2_vars = [name for name, spec in EUROSTAT_SOURCES.items() if spec["geo_level"] == "nuts2"]

print(nuts3_vars)
print(nuts2_vars)

In [ ]:
nuts3_frames = []
for var_name in nuts3_vars:
    df = load_eurostat_variable(var_name)
    nuts3_frames.append(df)

nuts3_eurostat = nuts3_frames[0]
for df in nuts3_frames[1:]:
    nuts3_eurostat = nuts3_eurostat.merge(df, on=["nuts3_id", "year"], how="outer")

del nuts3_frames
print(nuts3_eurostat.shape)
nuts3_eurostat.head()

In [ ]:
print("area nem-NaN évei:", sorted(nuts3_eurostat.loc[nuts3_eurostat["area"].notna(), "year"].unique()))
print("active_enterprises nem-NaN évei:", sorted(nuts3_eurostat.loc[nuts3_eurostat["active_enterprises"].notna(), "year"].unique()))
print(nuts3_eurostat.isna().sum())

In [ ]:
raw_check = pd.read_csv(RAW_EUROSTAT_DIR / "bd_hgnace_r.tsv", sep="\t")
print(raw_check.columns.tolist())

In [ ]:
sample_row = raw_check[
    (raw_check["freq,indic_sbs,nace_r2,geo\\TIME_PERIOD"].str.contains("ENT_NR")) &
    (raw_check["freq,indic_sbs,nace_r2,geo\\TIME_PERIOD"].str.contains("B-S_X_O_S94")) &
    (raw_check["freq,indic_sbs,nace_r2,geo\\TIME_PERIOD"].str.contains("CZ010"))
]
print(sample_row)

In [ ]:
osm_all = pd.read_parquet(INTERIM_AGGREGATED_DIR / "nuts3_osm_metrics_all.parquet")

nuts3_master = nuts3_eurostat.merge(osm_all, on="nuts3_id", how="left")

nuts3_master = nuts3_master.merge(
    nuts3_gdf[["nuts3_id", "geometry"]],
    on="nuts3_id",
    how="left",
)
nuts3_master["country_code"] = nuts3_master["nuts3_id"].str[:2]

print(nuts3_master.shape)
nuts3_master.head()

In [ ]:
nuts3_master = gpd.GeoDataFrame(nuts3_master, geometry="geometry", crs=nuts3_gdf.crs)

nuts3_master.to_parquet(NUTS3_MASTER_TABLE_PATH)
print(f"Mentve -> {NUTS3_MASTER_TABLE_PATH.relative_to(PROJECT_ROOT)} ({nuts3_master.shape})")

In [ ]:
nuts2_master = ai_df.merge(
    nuts2_gdf[["nuts2_id", "geometry"]],
    on="nuts2_id",
    how="left",
)
nuts2_master["country_code"] = nuts2_master["nuts2_id"].str[:2]

nuts2_master = gpd.GeoDataFrame(nuts2_master, geometry="geometry", crs=nuts2_gdf.crs)

print(nuts2_master.shape)
nuts2_master.head()

In [ ]:
NUTS2_AI_ADOPTION_TABLE_PATH = PROCESSED_DIR / "nuts2_ai_adoption_table.parquet"
nuts2_master.to_parquet(NUTS2_AI_ADOPTION_TABLE_PATH)
print(f"Mentve -> {NUTS2_AI_ADOPTION_TABLE_PATH.relative_to(PROJECT_ROOT)} ({nuts2_master.shape})")